# RAG Pipeline for Resume Retrieval (Semantic Chunking)

This notebook implements a RAG (Retrieval-Augmented Generation) pipeline using ChromaDB to retrieve relevant context from resume data. It uses **semantic chunking** — splitting by logical resume sections (Contact, Skills, Patents, Experience per company, Education) instead of fixed character counts.

In [ ]:
import re
import chromadb

_collection = None

# Section headers commonly found in LinkedIn PDF exports
_SECTION_HEADERS = [
    "Contact",
    "Top Skills",
    "Patents",
    "Summary",
    "Experience",
    "Education",
]

# Known company names / role markers that indicate a new experience block
_COMPANY_MARKERS = [
    "Nebula.io",
    "Simplified.Travel",
    "GQR Global Markets",
    "Wynden Stark",
    "untapt",
    "JPMorgan Chase",
    "JPMorgan",
    "Cygnifi",
    "IBM",
]

In [ ]:
def _clean_page_markers(text):
    """Remove LinkedIn PDF page markers like '  Page 1 of 5   '."""
    try:
        return re.sub(r'\s*Page \d+ of \d+\s*', '\n', text)
    except Exception:
        return text


def _semantic_chunk_linkedin(text):
    """
    Split LinkedIn PDF text into logical sections based on resume structure.
    Falls back to paragraph-based splitting if no section headers are found.

    Returns:
        List of (chunk_text, section_label) tuples.
    """
    try:
        if not text:
            return []

        text = _clean_page_markers(text)

        # Build a combined pattern from section headers and company markers
        all_markers = _SECTION_HEADERS + _COMPANY_MARKERS
        # Sort by length descending so longer markers match first
        all_markers_sorted = sorted(all_markers, key=len, reverse=True)
        # Escape for regex and join
        pattern_parts = [re.escape(m) for m in all_markers_sorted]
        split_pattern = r'(?=(?:^|\n)\s*(?:' + '|'.join(pattern_parts) + r'))'

        sections = re.split(split_pattern, text)
        chunks = []

        for section in sections:
            section = section.strip()
            if not section:
                continue

            # Determine the section label from the first line
            first_line = section.split('\n')[0].strip()
            label = "general"
            for marker in all_markers_sorted:
                if marker.lower() in first_line.lower():
                    label = marker.lower().replace(" ", "_").replace(".", "")
                    break

            chunks.append((section, label))

        if not chunks:
            # Fallback: split by double newlines (paragraphs)
            paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
            chunks = [(p, "general") for p in paragraphs]

        return chunks

    except Exception as e:
        print(f"Error in semantic chunking: {e}")
        return []

In [ ]:
def initialize_rag(linkedin_text, summary_text):
    """
    Initialize the RAG pipeline: semantically chunk documents and store in ChromaDB.
    Uses ChromaDB's built-in default embedding function (runs locally, no API needed).

    LinkedIn text is split by logical resume sections (contact, skills, patents,
    summary, each company experience, education).

    Args:
        linkedin_text: Full text extracted from LinkedIn PDF.
        summary_text: Full text from summary.txt.

    Returns:
        Number of chunks stored in the vector store.
    """
    global _collection

    try:
        linkedin_chunks = _semantic_chunk_linkedin(linkedin_text)

        all_chunks = []
        all_ids = []
        all_metadatas = []

        for i, (chunk_text, section_label) in enumerate(linkedin_chunks):
            all_chunks.append(chunk_text)
            all_ids.append(f"linkedin_{section_label}_{i}")
            all_metadatas.append({"source": "linkedin", "section": section_label})

        # Add summary as a single chunk
        if summary_text and summary_text.strip():
            all_chunks.append(summary_text.strip())
            all_ids.append("summary_0")
            all_metadatas.append({"source": "summary", "section": "personal_summary"})

        chroma_client = chromadb.Client()

        try:
            chroma_client.delete_collection("resume_rag")
        except Exception:
            pass

        _collection = chroma_client.create_collection(name="resume_rag")

        _collection.add(
            documents=all_chunks,
            ids=all_ids,
            metadatas=all_metadatas
        )

        print(f"RAG initialized: {_collection.count()} semantic chunks stored")
        for i, (chunk, label) in enumerate(linkedin_chunks):
            print(f"  [{i}] {label}: {chunk[:80]}...")
        print(f"  [+] personal_summary: {summary_text[:80].strip()}...")
        return _collection.count()

    except Exception as e:
        print(f"Error initializing RAG: {e}")
        return 0

In [ ]:
def retrieve_context(query, top_k=3):
    """
    Retrieve the most relevant chunks for a user query.

    Args:
        query: The user's question/message.
        top_k: Number of top chunks to retrieve.

    Returns:
        A string of the most relevant chunks joined by separators.
    """
    try:
        if _collection is None:
            raise ValueError("RAG not initialized. Call initialize_rag() first.")

        results = _collection.query(
            query_texts=[query],
            n_results=top_k
        )

        if results and results["documents"]:
            retrieved_text = "\n\n---\n\n".join(results["documents"][0])
            return retrieved_text
        return ""

    except Exception as e:
        print(f"Error retrieving context: {e}")
        return ""

In [ ]:
# TODO: Delete this cell - no longer needed (old _get_embedding and retrieve_context with Gemini embeddings)